In [1]:
#reset không gian làm việc
!rm -rf /content/*


'rm' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
# Tải toàn bộ kho lưu trữ
!git clone https://github.com/congnghia0609/ntc-scv.git

Cloning into 'ntc-scv'...
Updating files:  36% (4/11)
Updating files:  45% (5/11)
Updating files:  54% (6/11)
Updating files:  63% (7/11)
Updating files:  72% (8/11)
Updating files:  81% (9/11)
Updating files:  90% (10/11)
Updating files: 100% (11/11)
Updating files: 100% (11/11), done.


In [3]:
%pip install tensorflow==2.16.2 keras==3.14.1 matplotlib


  Obtaining dependency information for keras from https://files.pythonhosted.org/packages/02/03/184267c1d09783dd070f1ddfd0d4beb7503139dfc7bd75b422867cf282fd/keras-3.14.1-py3-none-any.whl.metadata
  Obtaining dependency information for tensorflow from https://files.pythonhosted.org/packages/7d/0d/4ee4bc074597b41c9c00dc97b4418ef1eb8736fe9186ffdb3961efdfb730/tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for matplotlib from https://files.pythonhosted.org/packages/04/a1/4571fc46e7702de8d0c2dc54ad1b2f8e29328dea3ee90831181f7353d93c/matplotlib-3.10.9-cp312-cp312-win_amd64.whl.metadata
     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     -------------------------------------- - 51.2/52.8 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 52.8/52.8 kB 906.3 kB/s eta 0:00:00
  Obtaining dependency information for absl-py from https://files.pythonhosted.org/packages/18/a6/907a406bb7d359e6a63f99c313846d9eec4f7e6f7437

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import glob
import zipfile

# Giải nén dữ liệu từ file zip
def unzip_data(zip_file, extract_to):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

# Giải nén file zip dữ liệu huấn luyện và kiểm tra
unzip_data('ntc-scv/data/data_train.zip', 'ntc-scv/data/data_train')
unzip_data('ntc-scv/data/data_test.zip', 'ntc-scv/data/data_test')


In [ ]:
# Cài đặt thư viện
%pip install pyvi

from sentiment_preprocessing import clean_text, load_stopwords


In [ ]:
# Set the data paths
train_dir = 'ntc-scv/data/data_train/data_train/train'
train_add_dir = 'ntc-scv/data/data_train/data_train/test'
test_dir = 'ntc-scv/data/data_test/data_test/test'
from pyvi import ViTokenizer
import os

# Hàm tải dữ liệu từ thư mục
def load_data_from_dir_pyvi(directory):
    texts = []
    labels = []

    for label_type in ['neg', 'pos']:
        dir_name = os.path.join(directory, label_type)
        for fname in os.listdir(dir_name):
            if fname.endswith('.txt'):
                with open(os.path.join(dir_name, fname), encoding='utf-8') as f:
                    text = f.read()
                    texts.append(text)
                if label_type == 'neg':
                    labels.append(0)
                else:
                    labels.append(1)
    return texts, labels

# Tải dữ liệu train và test
train_texts, train_labels = load_data_from_dir_pyvi(train_dir)
train_add_texts, train_add_labels = load_data_from_dir_pyvi(train_add_dir)
test_texts, test_labels = load_data_from_dir_pyvi(test_dir)
train_texts.extend(train_add_texts)
train_labels.extend(train_add_labels)
print(train_texts[0])
train_labels[0]


In [ ]:
print(len(train_texts))
print(len(test_texts))
#

In [ ]:
stopwords = load_stopwords()

# Dung chung preprocessing voi API de tokenizer.pkl va model.keras khop inference.
# Van bo stopwords, nhung neu cau ngan bi xoa sach thi fallback ve cau goc da normalize.
def clean_text_for_training(text):
    cleaned_text, _ = clean_text(text, stopwords)
    return cleaned_text


In [ ]:
train_texts = [clean_text_for_training(text) for text in train_texts]
test_texts = [clean_text_for_training(text) for text in test_texts]
train_texts[0]

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pickle
# Thông số cho mô hình
vocab_size = 20000   # Số lượng từ vựng tối đa trong từ điển. 
                     # Giảm số lượng từ giúp giảm kích thước mô hình và tiết kiệm tài nguyên tính toán,
                     # nhưng vẫn đủ để giữ lại các từ quan trọng nhất.

max_length = 300     # Độ dài tối đa của mỗi câu đầu vào. Các câu ngắn hơn sẽ được padding, 
                     # còn câu dài hơn sẽ được cắt ngắn. Độ dài này giúp cân bằng giữa độ chi tiết
                     # và hiệu quả tính toán.

embedding_dim = 128  # Kích thước vector embedding.

# Tokenizer cho văn bản
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(train_texts)

# Đường dẫn để lưu tokenizer
tokenizer_path = 'tokenizer.pkl'

# Lưu tokenizer vào file
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
word_index = tokenizer.word_index




In [ ]:
word_index

In [ ]:
# Chuyển đổi văn bản thành chuỗi số
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

# Padding chuỗi để đảm bảo cùng độ dài
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

# Chuyển đổi nhãn thành numpy arrays
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print(train_padded[0])
print(train_labels[0])

In [ ]:
# tensorflow-addons is not used by this notebook.


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Embedding, Dropout, GlobalMaxPooling1D, Input, Attention

vocab_size = 20000  # Kích thước từ vựng
embedding_dim = 128  # Kích thước vector nhúng
max_length = 300  # Độ dài tối đa của chuỗi đầu vào

# Xây dựng mô hình BiLSTM với lớp Attention
input_layer = Input(shape=(max_length,))  # Kích thước đầu vào
embedding_layer = Embedding(vocab_size, embedding_dim)(input_layer)

# Lớp BiLSTM
bi_lstm_output = Bidirectional(LSTM(32, return_sequences=True))(embedding_layer)

# Lớp Attention
attention_output = Attention()([bi_lstm_output, bi_lstm_output])  # Attention dùng cho đầu ra của BiLSTM

# Global Max Pooling
pooled_output = GlobalMaxPooling1D()(attention_output)

# Lớp Dense
dense_output1 = Dense(64, activation='relu')(pooled_output)
dropout_output = Dropout(0.5)(dense_output1)
dense_output2 = Dense(32, activation='relu')(dropout_output)

# Lớp đầu ra với sigmoid cho phân loại nhị phân
output_layer = Dense(1, activation='sigmoid')(dense_output2)

# Tạo mô hình
model = tf.keras.Model(inputs=input_layer, outputs=output_layer)

# Biên dịch mô hình
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Xây dựng mô hình
model.build(input_shape=(None, max_length))
model.summary()


In [ ]:
import tensorflow as tf
print(tf.__version__)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Tạo EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Theo dõi 'val_loss' để quyết định khi nào dừng
    patience=4,          # Số epoch không cải thiện trước khi dừng (ở đây là 4)
    restore_best_weights=True  # Khôi phục trọng số tốt nhất sau khi dừng
)

# Huấn luyện mô hình với EarlyStopping
history = model.fit(
    train_padded,
    train_labels,
    epochs=15,
    validation_data=(test_padded, test_labels),
    batch_size=64,
    callbacks=[early_stopping]  # Thêm EarlyStopping vào callbacks
)


In [ ]:
import matplotlib.pyplot as plt
# Lấy thông tin lịch sử huấn luyện
history_dict = history.history
accuracy = history_dict['accuracy']
val_accuracy = history_dict['val_accuracy']
loss = history_dict['loss']
val_loss = history_dict['val_loss']
epochs = range(1, len(accuracy) + 1)

# Vẽ đồ thị cho Training và Validation Loss
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs, loss, 'bo-', label='Training loss')  # Đường màu xanh
plt.plot(epochs, val_loss, 'ro-', label='Validation loss')  # Đường màu đỏ
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Vẽ đồ thị cho Training và Validation Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, accuracy, 'bo-', label='Training accuracy')  # Đường màu xanh
plt.plot(epochs, val_accuracy, 'ro-', label='Validation accuracy')  # Đường màu đỏ
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Hiển thị đồ thị
plt.tight_layout()
plt.show()

In [ ]:
model.save("phan_tich_cam_xuc.keras")
